In [2]:
import subprocess
result = subprocess.run(['grep', 'avx2', '/proc/cpuinfo'], capture_output=True, text=True)
print(result.stdout[:500] if result.stdout else "No AVX2")

flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm rdseed adx smap xsaveopt arat md_clear arch_capabilities
flags		: fpu vme d


In [4]:
!apt-get update -qq
!apt-get install -y cmake clang build-essential libssl-dev libcurl4-openssl-dev -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
!clang --version

Ubuntu clang version 14.0.0-1ubuntu1.1
Target: x86_64-pc-linux-gnu
Thread model: posix
InstalledDir: /usr/bin


In [6]:
!git clone https://github.com/endee-io/endee
%cd endee

fatal: destination path 'endee' already exists and is not an empty directory.
/content/endee


In [7]:
!CC=clang CXX=clang++ ./install.sh --release --avx2

[INFO] Selected Build Mode: release
[INFO] Selected CPU Flag:   avx2
[INFO] Identifying environment...
[INFO] Environment: OS_FAMILY=linux, ARCH=x86_64, DISTRO=ubuntu (22.04)
--- Detected Ubuntu (Version: 22.04) ---
[INFO] ubuntu 22.04: Clang-19 already detected. Skipping LLVM repo setup.
[INFO] Package already installed (apt): cmake
[INFO] Package already installed (apt): clang-19
[INFO] Package already installed (apt): build-essential
[INFO] Package already installed (apt): libssl-dev
[INFO] Package already installed (apt): libcurl4-openssl-dev
[INFO] Package already installed (apt): unzip
[INFO] Package already installed (apt): curl
[INFO] Package already installed (apt): git
[INFO] All apt packages already present.
[INFO] Mode: Release
[INFO] CPU Target: avx2
[INFO] Running cmake with flags: -DUSE_AVX2=ON
-- The C compiler identification is Clang 14.0.0
-- The CXX compiler identification is Clang 14.0.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Che

In [9]:
import subprocess
import time

process = subprocess.Popen(
    ['./run.sh'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)
print("Endee started!")

Endee started!


In [8]:
import subprocess
import time
import os

env = os.environ.copy()
env['NDD_DATA_DIR'] = './data'
env['NDD_AUTH_TOKEN'] = ''
env['NDD_NUM_THREADS'] = '2'

# Try different port variables
for port_var in ['NDD_PORT', 'NDD_SERVER_PORT', 'SERVER_PORT', 'PORT']:
    env[port_var] = '7070'

process = subprocess.Popen(
    ['./build/ndd-avx2'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=env
)
time.sleep(8)

stderr = process.stderr.read1(3000).decode('utf-8', errors='ignore')
print(stderr)
print("Running:", process.poll() is None)

INFO_1718: -/-: AVX2 is supported and usable
INFO: -/-: SERVER_ID: unknown
INFO: -/-: SERVER_PORT: 7070
INFO: -/-: DATA_DIR: ./data
INFO: -/-: NUM_PARALLEL_INSERTS: 4
INFO: -/-: NUM_RECOVERY_THREADS: 16
INFO: -/-: MAX_MEMORY_GB: 24
INFO: -/-: ENABLE_DEBUG_LOG: 1
INFO: -/-: AUTH_TOKEN: 
INFO: -/-: AUTH_ENABLED: 0
INFO: -/-: DEFAULT_USERNAME: endee
INFO: -/-: DEFAULT_SERVER_TYPE: OSS
INFO: -/-: DEFAULT_DATA_DIR: /mnt/data
INFO: -/-: DEFAULT_MAX_ACTIVE_INDICES: 64
INFO: -/-: DEFAULT_MAX_ELEMENTS: 100000
INFO: -/-: DEFAULT_MAX_ELEMENTS_INCREMENT: 100000
INFO: -/-: DEFAULT_MAX_ELEMENTS_INCREMENT_TRIGGER: 50000
INFO_1005: -/-: Starting the server
INFO_1102: -/-: Authentication disabled; running in open mode
INFO_1006: -/-: Created auth manager
INFO_1007: -/-: Created index manager
INFO_1008: -/-: Using: 4 server threads.
INFO_2011: -/-: Autosave thread started
(2026-03-24 13:54:23) [INFO    ] Crow/master server is running at http://0.0.0.0:7070 using 4 threads
(2026-03-24 13:54:23) [INFO    

In [10]:
import requests

r = requests.get("http://localhost:7070/api/v1/index/list")
print("Status:", r.status_code)
print("Response:", r.text)

Status: 200
Response: {"indexes":[{"name":"studymind_index","total_elements":376,"dimension":384,"sparse_model":"None","space_type":"cosine","precision":"int8","checksum":-1,"M":16,"created_at":1774359013}]}


In [11]:
!pip install fastapi uvicorn python-multipart sentence-transformers pypdf groq openai python-dotenv endee requests -q

In [12]:
import os
os.makedirs('/content/studymind/backend', exist_ok=True)
os.makedirs('/content/studymind/frontend', exist_ok=True)
print("✅ Folders created!")

✅ Folders created!


In [13]:
!pip install PyPDF2 pdfplumber

In [14]:
!pip install endee

In [15]:
!nohup endee start --port 7070 > endee_log.txt 2>&1 &

In [16]:
!curl http://localhost:7070/api/v1/index/list

{"indexes":[{"name":"studymind_index","total_elements":376,"dimension":384,"sparse_model":"None","space_type":"cosine","precision":"int8","checksum":-1,"M":16,"created_at":1774359013}]}

In [ ]:
env_content = """GROQ_API_KEY=your_groq_api_key
ENDEE_HOST=http://localhost:7070
ENDEE_API_KEY=
"""
with open('/content/studymind/backend/.env', 'w') as f:
    f.write(env_content)

print("✅ .env created!")

✅ .env created!


In [18]:
with open('/content/studymind/backend/main.py', 'w') as f:
    f.write('''from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
import os

from ingest import ingest_document, list_documents, delete_document
from study_chain import answer_question, generate_summary

app = FastAPI(title="StudyMind API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
FRONTEND_DIR = os.path.join(BASE_DIR, "..", "frontend")


# ---------------- UI ----------------
@app.get("/")
def serve_ui():
    return FileResponse(os.path.join(FRONTEND_DIR, "index.html"))


# ---------------- HEALTH ----------------
@app.get("/health")
def health():
    return {"status": "ok"}


# ---------------- UPLOAD ----------------
@app.post("/upload")
async def upload(file: UploadFile = File(...)):
    try:
        if file.content_type not in ["text/plain", "application/pdf"]:
            raise HTTPException(400, "Only .txt and .pdf supported")

        content = await file.read()

        result = ingest_document(content, file.filename, file.content_type)

        return {"message": "uploaded", "chunks": result.get("chunks_stored", 0)}

    except Exception as e:
        print("UPLOAD ERROR:", str(e))  #  shows real error in logs
        raise HTTPException(status_code=500, detail=str(e))

# ---------------- DOCUMENTS ----------------
@app.get("/documents")
def docs():
    return {"documents": list_documents()}


@app.delete("/documents/{filename}")
def delete(filename: str):
    delete_document(filename)
    return {"message": "deleted"}


# ---------------- ASK (RAG + GROQ) ----------------
@app.get("/ask")
def ask(query: str, top_k: int = 5):
    return answer_question(query, top_k)


# ---------------- SUMMARY (GROQ) ----------------
@app.get("/summary")
def summary(query: str):
    return generate_summary(query)


# ---------------- FIX ICON ERROR ----------------
@app.get("/favicon.ico")
async def favicon():
    return {}
''')

In [15]:
!pip install endee pypdf sentence-transformers

In [21]:
with open('/content/studymind/backend/ingest.py', 'w') as f:
    f.write('''import re, io, os, hashlib
from typing import List, Dict
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
load_dotenv()

EMBED_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 400
CHUNK_OVERLAP = 80

ENDEE_HOST = os.getenv("ENDEE_HOST", "http://localhost:7070")
ENDEE_API_KEY = os.getenv("ENDEE_API_KEY", "")
ENDEE_INDEX = "studymind_index"

_embedder = None
_endee_client = None
_endee_index = None
_uploaded_docs = set()

def _get_embedder():
    global _embedder
    if _embedder is None:
        print("Loading embedding model...")
        _embedder = SentenceTransformer(EMBED_MODEL)
        print("Model ready.")
    return _embedder

def _get_endee_index():
    global _endee_client, _endee_index

    if _endee_index is None:
        import requests
        from endee import Endee, Precision

        _endee_client = Endee(ENDEE_API_KEY) if ENDEE_API_KEY else Endee()
        _endee_client.set_base_url(f"{ENDEE_HOST}/api/v1")

        # list_indexes() returns raw strings, not objects
        try:
            resp = requests.get(f"{ENDEE_HOST}/api/v1/index/list")
            data = resp.json()
            existing_names = [idx["name"] for idx in data.get("indexes", [])]
        except Exception:
            existing_names = []

        if ENDEE_INDEX not in existing_names:
            _endee_client.create_index(
                name=ENDEE_INDEX,
                dimension=384,
                space_type="cosine",
                precision=Precision.INT8
            )

        _endee_index = _endee_client.get_index(name=ENDEE_INDEX)
        print("✅ Connected to Endee index")

    return _endee_index

def _extract_text(content, content_type):
    if content_type == "text/plain":
        return content.decode("utf-8", errors="ignore")

    if content_type == "application/pdf":
        import pypdf
        reader = pypdf.PdfReader(io.BytesIO(content))
        return "\\n".join(p.extract_text() or "" for p in reader.pages)

    raise ValueError(f"Unsupported type: {content_type}")

def _chunk_text(text):
    text = re.sub(r'\\s+', ' ', text).strip()
    chunks, start = [], 0

    while start < len(text):
        chunk = text[start:start+CHUNK_SIZE].strip()
        if chunk:
            chunks.append(chunk)
        start += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks

def _chunk_id(filename, i):
    return hashlib.md5(f"{filename}__chunk_{i}".encode()).hexdigest()

def ingest_document(content, filename, content_type):
    text = _extract_text(content, content_type)

    if not text.strip():
        raise ValueError("Document is empty.")

    chunks = _chunk_text(text)

    embedder = _get_embedder()
    vectors = embedder.encode(chunks, show_progress_bar=True).tolist()

    index = _get_endee_index()

    items = []
    for i in range(len(chunks)):
        items.append({
            "id": _chunk_id(filename, i),
            "vector": vectors[i],
            "meta": {
                "text": chunks[i],
                "source": filename,
                "chunk_id": i
            }
        })

    # batch insert
    for i in range(0, len(items), 100):
        index.upsert(items[i:i+100])

    _uploaded_docs.add(filename)

    return {"chunks_stored": len(chunks)}

def list_documents():
    return sorted(_uploaded_docs)

def delete_document(filename):
    _uploaded_docs.discard(filename)
''')

In [22]:
with open('/content/studymind/backend/retriever.py', 'w') as f:
    f.write('''from ingest import _get_embedder, _get_endee_index

def semantic_search(query, top_k=5):
    index = _get_endee_index()
    embedder = _get_embedder()

    query_vec = embedder.encode([query])[0].tolist()

    results = index.query(vector=query_vec, top_k=top_k)

    output = []
    for r in results:
        meta = r.get("meta", {})

        output.append({
            "text": meta.get("text", ""),
            "source": meta.get("source", ""),
            "similarity": r.get("score", 0)
        })

    return output
''')

In [33]:
with open('/content/studymind/backend/study_chain.py', 'w') as f:
    f.write('''import os
from dotenv import load_dotenv
from retriever import semantic_search

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")


# ---------------- LLM CALL ----------------
def _call_llm(prompt):
    if GROQ_API_KEY:
        from groq import Groq
        client = Groq(api_key=GROQ_API_KEY)

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=1000
        )

        return response.choices[0].message.content.strip()

    return None


# ---------------- CONTEXT ----------------
def _build_context(chunks):
    return "\\n\\n".join([
        f"[Source: {c['source']}]\\n{c['text']}"
        for c in chunks
    ])

# ---------------- ASK (RAG) ----------------
def answer_question(question, top_k=5):
    chunks = semantic_search(question, top_k)

    if not chunks:
        return {
            "answer": "No documents uploaded.",
            "sources": []
        }

    context = _build_context(chunks)

    prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    answer = _call_llm(prompt)

    if not answer:
        answer = chunks[0]["text"][:300]

    return {
        "answer": answer,
        "sources": [
            {
                "source": c["source"],
                "score": c["similarity"]   # fixed: was c["score"]
            } for c in chunks
        ]
    }


# ---------------- SUMMARY ----------------
def generate_summary(query):
    chunks = semantic_search(query, 5)

    if not chunks:
        return {
            "summary": "No documents uploaded."
        }

    context = _build_context(chunks)

    prompt = f"""
Summarize the following content clearly:

{context}

Summary:
"""

    summary = _call_llm(prompt)

    if not summary:
        summary = context[:500]

    return {
        "summary": summary
    }
''')

print("✅ Clean study_chain.py ready!")

✅ Clean study_chain.py ready!


In [36]:
import importlib
import study_chain
importlib.reload(study_chain)

<module 'study_chain' from '/content/studymind/backend/study_chain.py'>

In [24]:
print(os.listdir('/content/studymind/backend'))

['.env', 'retriever.py', 'study_chain.py', 'ingest.py', 'log.txt', 'main.py', '__pycache__']


In [25]:
html_content = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>StudyMind — AI Study Assistant</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:wght@300;400;500;600&display=swap" rel="stylesheet"/>
<style>
  :root {
    --bg: #0d0f14;
    --surface: #161923;
    --surface2: #1e2433;
    --border: #2a3045;
    --accent: #7c6af7;
    --accent2: #a78bfa;
    --green: #34d399;
    --red: #f87171;
    --text: #e8eaf0;
    --muted: #6b7595;
    --dim: #3d4561;
  }

  * { box-sizing: border-box; margin: 0; padding: 0; }

  body {
    background: var(--bg);
    color: var(--text);
    font-family: 'DM Sans', sans-serif;
    min-height: 100vh;
    display: flex;
    flex-direction: column;
  }

  body::before {
    content: '';
    position: fixed;
    inset: 0;
    background-image: url("data:image/svg+xml,%3Csvg viewBox='0 0 256 256' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='noise'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='0.9' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23noise)' opacity='0.04'/%3E%3C/svg%3E");
    pointer-events: none;
    z-index: 0;
  }

  .ambient {
    position: fixed;
    width: 600px; height: 600px;
    border-radius: 50%;
    background: radial-gradient(circle, rgba(124,106,247,0.07) 0%, transparent 70%);
    top: -200px; right: -200px;
    pointer-events: none;
    z-index: 0;
  }

  header {
    position: relative;
    z-index: 1;
    padding: 28px 40px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    border-bottom: 1px solid var(--border);
  }

  .logo { display: flex; align-items: center; gap: 12px; }

  .logo-icon {
    width: 38px; height: 38px;
    background: linear-gradient(135deg, var(--accent), var(--accent2));
    border-radius: 10px;
    display: flex; align-items: center; justify-content: center;
    font-size: 18px;
  }

  .logo-text { font-family: 'DM Serif Display', serif; font-size: 22px; letter-spacing: -0.3px; }
  .logo-text span { color: var(--accent2); font-style: italic; }

  .status-pill {
    display: flex; align-items: center; gap: 7px;
    padding: 6px 14px;
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 100px;
    font-size: 12px; color: var(--muted);
  }

  .status-dot {
    width: 7px; height: 7px;
    border-radius: 50%;
    background: var(--green);
    box-shadow: 0 0 8px var(--green);
    animation: pulse 2s infinite;
  }

  @keyframes pulse { 0%,100%{opacity:1} 50%{opacity:.4} }

  main {
    position: relative;
    z-index: 1;
    flex: 1;
    max-width: 1000px;
    margin: 0 auto;
    width: 100%;
    padding: 40px 24px 60px;
    display: flex;
    flex-direction: column;
    gap: 28px;
  }

  .section-label {
    display: flex; align-items: center; gap: 8px;
    font-size: 11px; font-weight: 600;
    letter-spacing: 1.5px; text-transform: uppercase;
    color: var(--muted);
    margin-bottom: 14px;
  }
  .section-label::after { content:''; flex:1; height:1px; background:var(--border); }

  .card {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 16px;
    padding: 24px;
    transition: border-color 0.2s;
  }
  .card:hover { border-color: var(--dim); }

  .upload-zone {
    border: 2px dashed var(--border);
    border-radius: 12px;
    padding: 36px;
    text-align: center;
    cursor: pointer;
    transition: all 0.2s;
    position: relative;
    overflow: hidden;
  }
  .upload-zone:hover, .upload-zone.drag-over {
    border-color: var(--accent);
    background: rgba(124,106,247,0.05);
  }
  .upload-zone input[type="file"] {
    position: absolute; inset: 0;
    opacity: 0; cursor: pointer;
    width: 100%; height: 100%;
  }
  .upload-icon { font-size: 36px; margin-bottom: 12px; display: block; }
  .upload-text { font-size: 15px; font-weight: 500; color: var(--text); margin-bottom: 4px; }
  .upload-subtext { font-size: 12px; color: var(--muted); }

  .upload-actions { display: flex; gap: 10px; margin-top: 16px; }

  .file-badge {
    display: none; align-items: center; gap: 8px;
    padding: 8px 14px;
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 8px;
    font-size: 13px; color: var(--text);
    margin-top: 12px;
  }
  .file-badge.show { display: flex; }

  .btn {
    display: inline-flex; align-items: center; gap: 7px;
    padding: 10px 20px;
    border-radius: 10px;
    font-family: 'DM Sans', sans-serif;
    font-size: 13px; font-weight: 500;
    cursor: pointer; border: none;
    transition: all 0.15s;
    letter-spacing: 0.1px;
  }
  .btn-primary {
    background: linear-gradient(135deg, var(--accent), var(--accent2));
    color: #fff;
    box-shadow: 0 4px 16px rgba(124,106,247,0.3);
  }
  .btn-primary:hover { transform: translateY(-1px); box-shadow: 0 6px 20px rgba(124,106,247,0.4); }
  .btn-primary:active { transform: translateY(0); }
  .btn-secondary {
    background: var(--surface2); color: var(--text);
    border: 1px solid var(--border);
  }
  .btn-secondary:hover { border-color: var(--dim); background: #262d40; }
  .btn:disabled { opacity: 0.4; cursor: not-allowed; transform: none !important; }

  .query-input {
    width: 100%;
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 12px;
    padding: 16px 20px;
    font-family: 'DM Sans', sans-serif;
    font-size: 15px; color: var(--text);
    outline: none; resize: none;
    min-height: 80px;
    transition: border-color 0.2s;
    line-height: 1.5;
  }
  .query-input::placeholder { color: var(--muted); }
  .query-input:focus {
    border-color: var(--accent);
    box-shadow: 0 0 0 3px rgba(124,106,247,0.12);
  }

  .query-actions {
    display: flex; gap: 10px;
    margin-top: 12px;
    flex-wrap: wrap; align-items: center;
  }

  .docs-list { display: flex; flex-direction: column; gap: 8px; }

  .doc-item {
    display: flex; align-items: center;
    justify-content: space-between;
    padding: 12px 16px;
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 10px;
    font-size: 13px;
    animation: fadeIn 0.3s ease;
  }
  .doc-item-left { display: flex; align-items: center; gap: 10px; color: var(--text); }
  .doc-icon { color: var(--accent2); font-size: 16px; }
  .doc-del {
    background: none; border: none;
    color: var(--muted); cursor: pointer;
    font-size: 16px; padding: 4px;
    border-radius: 6px;
    transition: all 0.15s; line-height: 1;
  }
  .doc-del:hover { color: var(--red); background: rgba(248,113,113,0.1); }

  .empty-docs {
    text-align: center; padding: 28px;
    color: var(--muted); font-size: 13px;
    border: 1px dashed var(--border);
    border-radius: 10px;
  }

  .output-area { min-height: 100px; }

  .output-box {
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 12px;
    padding: 20px;
    font-size: 14px; line-height: 1.75;
    color: var(--text);
    white-space: pre-wrap; word-break: break-word;
    animation: fadeIn 0.3s ease;
  }

  .output-placeholder {
    color: var(--muted); font-size: 14px;
    text-align: center; padding: 40px;
    border: 1px dashed var(--border);
    border-radius: 12px;
  }

  .sources-list { display: flex; flex-wrap: wrap; gap: 6px; margin-top: 14px; }
  .source-chip {
    display: inline-flex; align-items: center; gap: 5px;
    padding: 4px 10px;
    background: rgba(124,106,247,0.12);
    border: 1px solid rgba(124,106,247,0.25);
    border-radius: 100px;
    font-size: 11px; color: var(--accent2);
  }

  #toast {
    position: fixed; bottom: 28px; right: 28px;
    z-index: 999;
    display: flex; flex-direction: column; gap: 8px;
  }
  .toast-item {
    padding: 12px 18px; border-radius: 10px;
    font-size: 13px; font-weight: 500;
    max-width: 320px;
    animation: slideUp 0.3s ease;
    display: flex; align-items: center; gap: 8px;
  }
  .toast-success { background: rgba(52,211,153,0.15); border:1px solid rgba(52,211,153,0.3); color:var(--green); }
  .toast-error   { background: rgba(248,113,113,0.15); border:1px solid rgba(248,113,113,0.3); color:var(--red); }
  .toast-info    { background: rgba(124,106,247,0.15); border:1px solid rgba(124,106,247,0.3); color:var(--accent2); }

  .spinner {
    width: 16px; height: 16px;
    border: 2px solid rgba(255,255,255,0.3);
    border-top-color: #fff;
    border-radius: 50%;
    animation: spin 0.7s linear infinite;
  }
  .output-loading {
    display: flex; align-items: center; gap: 12px;
    padding: 28px;
    border: 1px solid var(--border); border-radius: 12px;
    color: var(--muted); font-size: 14px;
  }
  .output-loading .spinner {
    border-color: rgba(124,106,247,0.3);
    border-top-color: var(--accent);
  }

  .top-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }

  @media (max-width: 700px) {
    .top-grid { grid-template-columns: 1fr; }
    header { padding: 18px 20px; }
    main { padding: 24px 16px 40px; }
    .upload-zone { padding: 24px; }
    .query-actions { flex-direction: column; }
    .btn { justify-content: center; }
  }

  @keyframes fadeIn  { from{opacity:0;transform:translateY(6px)}  to{opacity:1;transform:translateY(0)} }
  @keyframes slideUp { from{opacity:0;transform:translateY(12px)} to{opacity:1;transform:translateY(0)} }
  @keyframes spin    { to{transform:rotate(360deg)} }

  .tab-row {
    display: flex; gap: 6px; padding: 4px;
    background: var(--surface2);
    border-radius: 10px; width: fit-content;
    border: 1px solid var(--border);
  }
  .tab {
    padding: 7px 16px; border-radius: 7px;
    font-size: 13px; font-weight: 500;
    cursor: pointer; border: none;
    background: transparent; color: var(--muted);
    font-family: 'DM Sans', sans-serif;
    transition: all 0.15s;
  }
  .tab.active {
    background: var(--surface); color: var(--text);
    box-shadow: 0 2px 8px rgba(0,0,0,0.3);
  }
</style>
</head>
<body>
<div class="ambient"></div>

<header>
  <div class="logo">
    <div class="logo-icon">📖</div>
    <div class="logo-text">Study<span>Mind</span></div>
  </div>
  <div class="status-pill">
    <span class="status-dot" id="statusDot"></span>
    <span id="statusLabel">Connecting…</span>
  </div>
</header>

<main>

  <div class="top-grid">
    <!-- UPLOAD -->
    <div class="card">
      <div class="section-label">Upload Document</div>
      <div class="upload-zone" id="uploadZone">
        <input type="file" id="fileInput" accept=".txt,.pdf" onchange="handleFileSelect(event)" />
        <span class="upload-icon">📄</span>
        <div class="upload-text">Drop a file or click to browse</div>
        <div class="upload-subtext">Supports .pdf and .txt</div>
      </div>
      <div class="file-badge" id="fileBadge">
        <span>📎</span>
        <span id="fileName"></span>
      </div>
      <div class="upload-actions">
        <button class="btn btn-primary" onclick="uploadFile()" id="uploadBtn" disabled>
          <span>⬆</span> Upload
        </button>
        <button class="btn btn-secondary" onclick="clearFile()">Clear</button>
      </div>
    </div>

    <!-- DOCUMENTS -->
    <div class="card">
      <div class="section-label">Uploaded Documents</div>
      <div class="docs-list" id="docsList">
        <div class="empty-docs">No documents yet.<br/>Upload one to get started.</div>
      </div>
    </div>
  </div>

  <!-- QUERY -->
  <div class="card">
    <div class="section-label">Ask or Summarize</div>
    <textarea class="query-input" id="queryInput"
      placeholder="E.g. What are the key concepts? or Summarize the introduction…"
      rows="3"></textarea>
    <div class="query-actions">
      <div class="tab-row">
        <button class="tab active" id="tabAsk" onclick="setMode('ask')">Ask Question</button>
        <button class="tab" id="tabSummarize" onclick="setMode('summarize')">Summarize</button>
      </div>
      <button class="btn btn-primary" onclick="runQuery()" id="queryBtn">
        <span>✦</span> <span id="queryBtnText">Ask</span>
      </button>
    </div>
  </div>

  <!-- OUTPUT -->
  <div class="card output-area">
    <div class="section-label">Answer</div>
    <div id="outputContainer">
      <div class="output-placeholder">Your answer will appear here…</div>
    </div>
  </div>

</main>

<div id="toast"></div>

<script>
  // ── Injected by Python — do not edit this line manually ──
  const API = "https://tomeka-humuslike-bioclimatologically.ngrok-free.dev"

  let selectedFile = null;
  let mode = 'ask';

  // ── HEALTH CHECK ──────────────────────────────
  async function checkHealth() {
    const label = document.getElementById('statusLabel');
    const dot   = document.getElementById('statusDot');
    try {
      const r = await fetch(`${API}/health`, { signal: AbortSignal.timeout(5000) });
      if (r.ok) {
        label.textContent = 'Connected';
        dot.style.background = 'var(--green)';
        dot.style.boxShadow  = '0 0 8px var(--green)';
      } else throw new Error();
    } catch {
      label.textContent = 'Offline';
      dot.style.background = 'var(--red)';
      dot.style.boxShadow  = '0 0 8px var(--red)';
    }
  }

  // ── FILE SELECT ───────────────────────────────
  function handleFileSelect(e) {
    const file = e.target.files[0];
    if (!file) return;
    selectedFile = file;
    document.getElementById('fileName').textContent = file.name;
    document.getElementById('fileBadge').classList.add('show');
    document.getElementById('uploadBtn').disabled = false;
  }

  function clearFile() {
    selectedFile = null;
    document.getElementById('fileInput').value = '';
    document.getElementById('fileBadge').classList.remove('show');
    document.getElementById('uploadBtn').disabled = true;
  }

  const zone = document.getElementById('uploadZone');
  zone.addEventListener('dragover', e => { e.preventDefault(); zone.classList.add('drag-over'); });
  zone.addEventListener('dragleave', () => zone.classList.remove('drag-over'));
  zone.addEventListener('drop', e => {
    e.preventDefault();
    zone.classList.remove('drag-over');
    const file = e.dataTransfer.files[0];
    if (!file) return;
    selectedFile = file;
    document.getElementById('fileName').textContent = file.name;
    document.getElementById('fileBadge').classList.add('show');
    document.getElementById('uploadBtn').disabled = false;
  });

  // ── UPLOAD ────────────────────────────────────
  async function uploadFile() {
    if (!selectedFile) return;
    const btn = document.getElementById('uploadBtn');
    btn.disabled = true;
    btn.innerHTML = '<span class="spinner"></span> Uploading…';

    const fd = new FormData();
    fd.append('file', selectedFile);

    try {
      const r = await fetch(`${API}/upload`, { method: 'POST', body: fd });
      const data = await r.json();
      if (!r.ok) throw new Error(data.detail || 'Upload failed');
      toast(`✓ Uploaded — ${data.chunks} chunks indexed`, 'success');
      clearFile();
      refreshDocs();
    } catch(err) {
      toast('Upload failed: ' + err.message, 'error');
    } finally {
      btn.disabled = false;
      btn.innerHTML = '<span>⬆</span> Upload';
    }
  }

  // ── DOCUMENTS ─────────────────────────────────
  async function refreshDocs() {
    try {
      const r = await fetch(`${API}/documents`);
      const data = await r.json();
      renderDocs(data.documents || []);
    } catch { /* silent */ }
  }

  function renderDocs(docs) {
    const el = document.getElementById('docsList');
    if (!docs.length) {
      el.innerHTML = '<div class="empty-docs">No documents yet.<br/>Upload one to get started.</div>';
      return;
    }
    el.innerHTML = docs.map(d => `
      <div class="doc-item">
        <div class="doc-item-left">
          <span class="doc-icon">${d.endsWith('.pdf') ? '📕' : '📄'}</span>
          <span>${escapeHtml(d)}</span>
        </div>
        <button class="doc-del" onclick="deleteDoc('${escapeHtml(d)}')" title="Remove">✕</button>
      </div>`).join('');
  }

  async function deleteDoc(name) {
    await fetch(`${API}/documents/${encodeURIComponent(name)}`, { method: 'DELETE' });
    toast(`Removed ${name}`, 'info');
    refreshDocs();
  }

  // ── MODE ──────────────────────────────────────
  function setMode(m) {
    mode = m;
    document.getElementById('tabAsk').classList.toggle('active', m === 'ask');
    document.getElementById('tabSummarize').classList.toggle('active', m === 'summarize');
    document.getElementById('queryBtnText').textContent = m === 'ask' ? 'Ask' : 'Summarize';
    document.getElementById('queryInput').placeholder = m === 'ask'
      ? 'Ask a question about your documents…'
      : 'What topic would you like summarized?';
  }

  // ── QUERY ─────────────────────────────────────
  async function runQuery() {
    const q = document.getElementById('queryInput').value.trim();
    if (!q) return toast('Enter a question first', 'error');

    const btn = document.getElementById('queryBtn');
    btn.disabled = true;
    btn.innerHTML = '<span class="spinner"></span> <span>Thinking…</span>';

    const container = document.getElementById('outputContainer');
    container.innerHTML = '<div class="output-loading"><div class="spinner"></div>Generating response…</div>';

    try {
      const ep = mode === 'ask'
        ? `/ask?query=${encodeURIComponent(q)}`
        : `/summary?query=${encodeURIComponent(q)}`;

      const r    = await fetch(`${API}${ep}`);
      const data = await r.json();
      if (!r.ok) throw new Error(data.detail || 'Request failed');

      const text    = mode === 'ask' ? data.answer : data.summary;
      const sources = mode === 'ask' ? (data.sources || []) : [];

      let html = `<div class="output-box">${escapeHtml(text)}</div>`;
      if (sources.length) {
        html += `<div class="sources-list">` +
          sources.map(s =>
            `<span class="source-chip">📎 ${escapeHtml(s.source)}
             <span style="opacity:.6">${(s.score * 100).toFixed(0)}%</span></span>`
          ).join('') + `</div>`;
      }
      container.innerHTML = html;
    } catch(err) {
      container.innerHTML = `<div class="output-box" style="color:var(--red)">Error: ${escapeHtml(err.message)}</div>`;
      toast('Request failed: ' + err.message, 'error');
    } finally {
      btn.disabled = false;
      btn.innerHTML = `<span>✦</span> <span id="queryBtnText">${mode === 'ask' ? 'Ask' : 'Summarize'}</span>`;
    }
  }

  document.getElementById('queryInput').addEventListener('keydown', e => {
    if (e.key === 'Enter' && !e.shiftKey) { e.preventDefault(); runQuery(); }
  });

  // ── TOAST ─────────────────────────────────────
  function toast(msg, type = 'info') {
    const el = document.createElement('div');
    el.className = `toast-item toast-${type}`;
    el.textContent = msg;
    document.getElementById('toast').appendChild(el);
    setTimeout(() => el.remove(), 3500);
  }

  function escapeHtml(s) {
    return String(s).replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');
  }

  // Auto-start
  checkHealth();
  refreshDocs();
</script>
</body>
</html>
"""

with open('/content/studymind/frontend/index.html', 'w') as f:
    f.write(html_content)

In [26]:
import subprocess
import threading
import time
import os

os.chdir('/content/studymind/backend')

def run_app():
    subprocess.run([
        'python', '-m', 'uvicorn', 'main:app',
        '--host', '0.0.0.0',
        '--port', '8000'
    ])

thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()
time.sleep(5)
print("✅ StudyMind started!")

✅ StudyMind started!


In [38]:
import subprocess, time, os

os.chdir('/content/studymind/backend')

# Kill any old process on 8000
subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True)
time.sleep(1)

# Start uvicorn fresh
subprocess.Popen(
    ['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=open('log.txt', 'w'),
    stderr=subprocess.STDOUT
)
time.sleep(5)
print("✅ Uvicorn started")

✅ Uvicorn started


In [27]:
!fuser -k 8000/tcp


8000/tcp:            66657


In [34]:
!pkill -f ngrok
!pkill -f uvicorn

In [35]:
!nohup uvicorn main:app --host 0.0.0.0 --port 8000 > log.txt 2>&1 &

In [36]:
import subprocess

subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
)

<Popen: returncode: None args: ['uvicorn', 'main:app', '--host', '0.0.0.0', ...>

In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token("Your_ngrok_authtoken")

public_url = ngrok.connect(8000)
print("🌐 Open this URL in your browser:")
print(public_url)

🌐 Open this URL in your browser:
NgrokTunnel: "https://tomeka-humuslike-bioclimatologically.ngrok-free.dev" -> "http://localhost:8000"


In [38]:
!cat log.txt


INFO:     Started server process [68408]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     2401:4900:892f:ff90:68e8:d64c:8bca:9980:0 - "GET / HTTP/1.1" 200 OK
INFO:     2401:4900:892f:ff90:68e8:d64c:8bca:9980:0 - "GET /health HTTP/1.1" 200 OK
INFO:     2401:4900:892f:ff90:68e8:d64c:8bca:9980:0 - "GET /documents HTTP/1.1" 200 OK
INFO:     2401:4900:892f:ff90:68e8:d64c:8bca:9980:0 - "GET /favicon.ico HTTP/1.1" 200 OK
Loading embedding model...
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1489.50it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Model ready.